# 04 · Sensitivity analysis: expert-curated predictors vs recursive feature elimination

Produces **Supplementary Table 5** and the numbers in the corresponding appendix text: the
collinearity of study site and ethnicity, recursive feature elimination with cross-validation (RFECV)
on the full candidate variable set, the stability of RFE across bootstrap resamples, and held-out
performance of the expert-curated (17 predictors) and RFECV-selected sets.

In [1]:
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import pearsonr
from sklearn.feature_selection import RFE, RFECV
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
RANDOM_STATE = 42
FINAL_PARAMS = {"colsample_bytree": 0.7, "learning_rate": 0.05, "max_depth": 8, "n_estimators": 400, "subsample": 0.8}
FEATURES = ["Age (y)", "Isotope", "Meal type", "Fe compound", "Fe dose", "Vitamin C dose",
                "Calcium dose", "Zinc dose ", "Polyphenol dose", "Phytic acid dose",
                "C-reactive protein", "Hep", "Ferritin", "Hb", "BMI", "BV", "Fe.circ"]
os.makedirs("../results", exist_ok=True)

raw = pd.read_excel("../data/pooled_dataset.xls")

## 1. Redundant and collinear variables excluded from the expert-curated set

In [2]:
sub = raw.dropna(subset=["BW", "BH", "BMI", "BV"])
print(f"corr(body weight, BMI) = {pearsonr(sub['BW'], sub['BMI'])[0]:.3f}; "
      f"corr(body weight, blood volume) = {pearsonr(sub['BW'], sub['BV'])[0]:.3f}")

ct = pd.crosstab(raw["Meal type"].fillna("Supplement"), raw["Meal or Supplement"])
print("\nMeal types assigned to more than one administration route:", int(((ct > 0).sum(axis=1) > 1).sum()),
      "(Meal or Supplement is determined by Meal type)")

print("\nEthnicity by study site (each site is a single ethnic cohort):")
print(pd.crosstab(raw["Ethnicity"], raw["Site"]))

for col, alt in [("sTfR", "Ferritin"), ("Alpha glycoprotein", "C-reactive protein")]:
    s = raw.dropna(subset=[col, alt])
    print(f"\n{col}: available in {raw[col].notna().mean():.0%} of measurements; "
          f"corr with {alt} = {pearsonr(s[col], s[alt])[0]:.2f}")

corr(body weight, BMI) = 0.868; corr(body weight, blood volume) = 0.966

Meal types assigned to more than one administration route: 0 (Meal or Supplement is determined by Meal type)

Ethnicity by study site (each site is a single ethnic cohort):
Site       Benin  Haiti  India  Lebanon  Mexico  Morocco  Switzerland  Taiwan  Thailand
Ethnicity                                                                              
Arab           0      0      0      114       0       84            0       0         0
Asian          0      0      0        0       0        0            0      70       156
Black         46     65      0        0       0        0            0       0         0
Caucasian      0      0      0        0       0        0         4137       0         0
Indian         0      0    158        0       0        0            0       0         0
Latin          0      0      0        0     228        0            0       0         0

sTfR: available in 30% of measurements; corr with

## 2. RFECV on the full candidate variable set

In [3]:
ID_COLS = ["Order", "Study", "Idold", "Gender", "MC.last"]
candidates = [c for c in raw.columns if c not in ID_COLS + ["Fe absorption (%)"]]
print(len(candidates), "candidate variables")

d = raw[candidates + ["Fe absorption (%)"]].copy()
d["Meal type"] = d["Meal type"].fillna("Supplement")
d["Isotope"] = d["Isotope"].fillna("Unknown")
for col in d.columns:
    if d[col].dtype == object:
        d[col] = d[col].astype("category").cat.codes
d = d[d[FEATURES + ["Fe absorption (%)"]].isnull().sum(axis=1) < 14]

y = np.log10(d["Fe absorption (%)"] + 1)
X = d[candidates]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)

rank_estimator = xgb.XGBRegressor(objective="reg:squarederror", n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rfecv = RFECV(rank_estimator, step=1, cv=5, scoring="r2", n_jobs=1, min_features_to_select=5)
rfecv.fit(X_train, y_train)
rfe_selected = list(X_train.columns[rfecv.support_])
print(f"RFECV selected {rfecv.n_features_} of {X_train.shape[1]} variables:")
print(rfe_selected)
for v in ["Ethnicity", "Site", "sTfR", "Alpha glycoprotein"]:
    print(f"  {v} selected: {v in rfe_selected}")

31 candidate variables


RFECV selected 29 of 31 variables:
['Age (y)', 'Ethnicity', 'Site', 'Isotope', 'Meal type', 'Meal or Supplement', 'Fe compound', 'Fe dose', 'Vitamin C dose', 'Calcium dose', 'Zinc dose ', 'Polyphenol dose', 'Phytic acid dose', 'Meat protein', 'Milk protein', 'C-reactive protein', 'Alpha glycoprotein', 'Hep', 'Ferritin', 'Hb', 'BW', 'BH', 'BMI', 'BV', 'Fe.circ', 'sTfR', 'MC.freq', 'MC.duration', 'MC.percentage']
  Ethnicity selected: True
  Site selected: True
  sTfR selected: True
  Alpha glycoprotein selected: True


## 3. Stability of RFE (17 variables) across 15 bootstrap resamples of the training set

In [4]:
n_runs = 15
rng = np.random.default_rng(RANDOM_STATE)
eth, site = 0, 0
for i in range(n_runs):
    seed = int(rng.integers(0, 100000))
    boot = rng.integers(0, len(X_train), size=len(X_train))
    est = xgb.XGBRegressor(objective="reg:squarederror", n_estimators=150, random_state=seed, n_jobs=-1)
    rfe = RFE(est, n_features_to_select=17, step=2).fit(X_train.iloc[boot], y_train.iloc[boot])
    sel = set(X_train.columns[rfe.support_])
    eth += "Ethnicity" in sel
    site += "Site" in sel
print(f"Ethnicity selected in {eth}/{n_runs} runs; study site selected in {site}/{n_runs} runs")

Ethnicity selected in 0/15 runs; study site selected in 13/15 runs


## 4. Supplementary Table 5: held-out performance of the two predictor sets

In [5]:
def eval_feature_set(features, label):
    m = xgb.XGBRegressor(objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=-1, **FINAL_PARAMS)
    m.fit(X_train[features], y_train)
    p = m.predict(X_test[features])
    return {"Predictor set": label, "n predictors": len(features),
            "Test RMSE": root_mean_squared_error(y_test, p), "Test MAE": mean_absolute_error(y_test, p),
            "Test R2": r2_score(y_test, p)}

supp_table5 = pd.DataFrame([
    eval_feature_set(FEATURES, "Expert-curated"),
    eval_feature_set(rfe_selected, "RFECV-selected"),
]).round(3)
print(supp_table5.to_string(index=False))
supp_table5.to_csv("../results/supp_table5_feature_selection.csv", index=False)

 Predictor set  n predictors  Test RMSE  Test MAE  Test R2
Expert-curated            17      0.206     0.160    0.757
RFECV-selected            29      0.205     0.158    0.759
